In [1]:
import pandas as pd

In [2]:
data = pd.read_csv('crop_yield.csv')

In [3]:
data.sample(5)

,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
8355,Ragi,2002,Autumn,Jharkhand,26659.73,32959,1041.2,2.523877e+06,6664.9325,1.048571
18450,Turmeric,1998,Autumn,Manipur,100.00,70,1855.0,9.880000e+03,29.0000,0.833333
15738,Tobacco,2017,Kharif,Mizoram,338.50,281,3865.8,5.329344e+04,128.6300,0.672500
13165,Tobacco,2013,Rabi,Gujarat,84085.00,149870,1006.5,1.214944e+07,22702.9500,1.714444
12192,Ginger,2011,Whole Year,Madhya Pradesh,9926.00,10714,852.6,1.662804e+06,3275.5800,0.946744


In [4]:
data.shape

(19689, 10)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Crop             19689 non-null  object 
 1   Crop_Year        19689 non-null  int64  
 2   Season           19689 non-null  object 
 3   State            19689 non-null  object 
 4   Area             19689 non-null  float64
 5   Production       19689 non-null  int64  
 6   Annual_Rainfall  19689 non-null  float64
 7   Fertilizer       19689 non-null  float64
 8   Pesticide        19689 non-null  float64
 9   Yield            19689 non-null  float64
dtypes: float64(5), int64(2), object(3)
memory usage: 1.5+ MB


In [6]:
data.describe()

,Crop_Year,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
count,19689.000000,1.968900e+04,1.968900e+04,19689.000000,1.968900e+04,1.968900e+04,19689.000000
mean,2009.127584,1.799266e+05,1.643594e+07,1437.755177,2.410331e+07,4.884835e+04,79.954009
std,6.498099,7.328287e+05,2.630568e+08,816.909589,9.494600e+07,2.132874e+05,878.306193
min,1997.000000,5.000000e-01,0.000000e+00,301.300000,5.417000e+01,9.000000e-02,0.000000
25%,2004.000000,1.390000e+03,1.393000e+03,940.700000,1.880146e+05,3.567000e+02,0.600000
50%,2010.000000,9.317000e+03,1.380400e+04,1247.600000,1.234957e+06,2.421900e+03,1.030000
75%,2015.000000,7.511200e+04,1.227180e+05,1643.700000,1.000385e+07,2.004170e+04,2.388889
max,2020.000000,5.080810e+07,6.326000e+09,6552.700000,4.835407e+09,1.575051e+07,21105.000000


In [7]:
X = data.drop('Yield',axis =1)
y = data['Yield']

In [8]:
from sklearn.preprocessing import OrdinalEncoder
cat_cols = ['State','Season','Crop']
le = OrdinalEncoder()
X[cat_cols] = le.fit_transform(X[cat_cols])

In [9]:
from sklearn.model_selection import train_test_split

In [10]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)


In [11]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(15751, 9)
(3938, 9)
(15751,)
(3938,)


In [12]:
from sklearn.feature_selection import mutual_info_regression

In [13]:
# Calculate Mutual Information scores
mi_scores = mutual_info_regression(X, y, random_state=42)
# Create a DataFrame
mi_df = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
})
# Sort features by MI score
mi_df = mi_df.sort_values(by='MI_Score', ascending=False)

print(mi_df)

           Feature  MI_Score
0             Crop  0.906335
5       Production  0.460155
3            State  0.377045
4             Area  0.254573
6  Annual_Rainfall  0.216861
7       Fertilizer  0.159913
2           Season  0.148879
8        Pesticide  0.136107
1        Crop_Year  0.003854


In [14]:
top_7 = mi_df.head(7)
print(top_7)

           Feature  MI_Score
0             Crop  0.906335
5       Production  0.460155
3            State  0.377045
4             Area  0.254573
6  Annual_Rainfall  0.216861
7       Fertilizer  0.159913
2           Season  0.148879


In [15]:
from sklearn.preprocessing import StandardScaler
selected_features = top_7['Feature'].tolist()
X = X[selected_features].copy()

# Scale only numerical features among the selected 7
numeric_cols = [
    'Production', 'Area', 'Annual_Rainfall', 'Fertilizer'
]

scale = StandardScaler()
X[numeric_cols] = scale.fit_transform(X[numeric_cols])

In [16]:
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

In [17]:
print(X_test_selected.shape)
print(X_train_selected.shape)

(3938, 7)
(15751, 7)


# KNeighborsRegressor

In [18]:
from sklearn.neighbors import KNeighborsRegressor

In [19]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train_selected,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [20]:
knn_pred = knn.predict(X_test_selected)

In [21]:
from sklearn.metrics import r2_score,mean_squared_error

In [22]:
print('R2_score:',r2_score(y_test,knn_pred))
print('MSE:',mean_squared_error(y_test,knn_pred))

R2_score: 0.8644686780936709
MSE: 108593.09805505403


Hyperparamter tuning

In [23]:
from sklearn.model_selection import GridSearchCV

In [24]:
param_grid = {
    'n_neighbors':[4,6,7,8,12,13,14],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan']
}

In [25]:
knn_grid = GridSearchCV(
    knn,param_grid,cv = 5, scoring = 'r2'
)

In [26]:
knn_grid.fit(X_train_selected,y_train)

,estimator,KNeighborsRegressor()
,param_grid,"{'metric': ['euclidean', 'manhattan'], 'n_neighbors': [4, 6, ...], 'weights': ['uniform', 'distance']}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_neighbors,4


In [27]:
print(knn_grid.best_params_)
print(knn_grid.best_score_)


{'metric': 'manhattan', 'n_neighbors': 4, 'weights': 'distance'}
0.9281814961192211


In [28]:
knn_model = knn_grid.best_estimator_

In [29]:
knn_pred2 = knn_model.predict(X_test_selected)

In [30]:
print('R2_score:',r2_score(y_test,knn_pred2))
print('MSE:',mean_squared_error(y_test,knn_pred2))

R2_score: 0.9117281261957391
MSE: 70726.94424211784


In [31]:
import pickle

with open("dtree_model.pkl", "wb") as file:
    pickle.dump({
        "model": knn_model,
        "encoder": le,
        "scaler": scale
    }, file)

# Decision Tree

In [19]:
from sklearn.tree import DecisionTreeRegressor

In [20]:
dt = DecisionTreeRegressor(max_depth = 4)

In [21]:
dt.fit(X_train_selected,y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,4
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [22]:
dt_pred = dt.predict(X_test_selected)

In [23]:
from sklearn.metrics import r2_score,mean_squared_error
print('R2_score:',r2_score(y_test,dt_pred))
print('MSE:',mean_squared_error(y_test,dt_pred))

R2_score: 0.816063960470713
MSE: 147376.88746419142


Hyperparameter Tuning

In [24]:
from sklearn.model_selection import GridSearchCV
import numpy as np

In [25]:
parameters = {
    'criterion': ['squared_error','absolute_error'],
    'splitter': ['best', 'random'],
    'max_depth': range(1,15),
    'max_features': ['sqrt', 'log2']
}

In [26]:
dt_grid = GridSearchCV(
    dt,parameters,cv = 5,scoring = 'r2'
)

In [27]:
dt_grid.fit(X_train_selected,y_train)

,estimator,DecisionTreeR...r(max_depth=4)
,param_grid,"{'criterion': ['squared_error', 'absolute_error'], 'max_depth': range(1, 20), 'max_features': ['sqrt', 'log2'], 'splitter': ['best', 'random']}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,criterion,'squared_error'


In [32]:
print(dt_grid.best_params_)
print(dt_grid.best_score_)


{'criterion': 'squared_error', 'max_depth': 18, 'max_features': 'sqrt', 'splitter': 'random'}
0.9729566507154038


In [33]:
dt_model = dt_grid.best_estimator_

In [34]:
dt_pred2 = dt_model.predict(X_test_selected)

In [35]:
print('R2_score:',r2_score(y_test,dt_pred2))
print('MSE:',mean_squared_error(y_test,dt_pred2))

R2_score: 0.9232905367542625
MSE: 61462.67996819362


In [37]:
import pickle

with open("dtree2_model.pkl", "wb") as file:
    pickle.dump({
        "model": dt_model,
        "encoder": le,
        "scaler": scale
    }, file)

In [39]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Predictions
train_pred = dt_model.predict(X_train_selected)
test_pred = dt_model.predict(X_test_selected)

# R2 scores
train_r2 = r2_score(y_train, train_pred)
test_r2 = r2_score(y_test, test_pred)

print("Training R2 Score:", train_r2)
print("Testing R2 Score:", test_r2)

# MAE
print("Training MAE:", mean_absolute_error(y_train, train_pred))
print("Testing MAE:", mean_absolute_error(y_test, test_pred))

Training R2 Score: 0.9999318091519233
Testing R2 Score: 0.9232905367542625
Training MAE: 1.34914148648688
Testing MAE: 12.785097748274824


In [40]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Get pruning path from existing model
path = dt_model.cost_complexity_pruning_path(X_train, y_train)

# Choose a small pruning value
ccp_alpha = np.quantile(path.ccp_alphas, 0.50)

# Create a pruned version using existing model parameters
regularized_model = DecisionTreeRegressor(
    **dt_model.get_params()
)

regularized_model.set_params(ccp_alpha=ccp_alpha)

# Fit the pruned model
regularized_model.fit(X_train, y_train)

# Predictions
train_pred = regularized_model.predict(X_train)
test_pred = regularized_model.predict(X_test)

# Evaluation
print("Training R2:", r2_score(y_train, train_pred))
print("Testing R2:", r2_score(y_test, test_pred))

print("Training MAE:", mean_absolute_error(y_train, train_pred))
print("Testing MAE:", mean_absolute_error(y_test, test_pred))

Training R2: 0.9972887406465619
Testing R2: 0.7988491949589871
Training MAE: 3.2342043972194032
Testing MAE: 23.02340239951378


# Multiple Linear Regression

In [44]:
from sklearn.linear_model import LinearRegression

In [45]:
lr = LinearRegression()
lr.fit(X_train_selected,y_train)


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [46]:
lr_pred = lr.predict(X_test_selected)

In [47]:
print('R2_score:',r2_score(y_test,lr_pred))
print('MSE:',mean_squared_error(y_test,lr_pred))

R2_score: 0.40064135178442306
MSE: 480230.04232779745


Hyperparameter Tuning

In [48]:
from sklearn.model_selection import GridSearchCV

In [49]:
parameters_lr = {
    'fit_intercept': [True, False],
    'positive': [True, False]
}

In [50]:
lr_grid = GridSearchCV(
    lr,parameters_lr,cv = 5, scoring = 'r2'
)

In [51]:
lr_grid.fit(X_train_selected,y_train)

,estimator,LinearRegression()
,param_grid,"{'fit_intercept': [True, False], 'positive': [True, False]}"
,scoring,'r2'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,fit_intercept,True


In [52]:
print(lr_grid.best_params_)
print(lr_grid.best_score_)

{'fit_intercept': True, 'positive': False}
0.31761157493181125


In [53]:
lr_model = lr_grid.best_estimator_

In [54]:
lr_pred2 = lr_model.predict(X_test_selected)

In [55]:
print('R2_score:',r2_score(y_test,lr_pred2))
print('MSE:',mean_squared_error(y_test,lr_pred2))

R2_score: 0.40064135178442306
MSE: 480230.04232779745


# Polynomial Linear Regression

In [56]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [57]:
pipe = Pipeline([(
    'poly',PolynomialFeatures(degree = 3,include_bias = False)),
                 ("regressor", LinearRegression())])

In [58]:
pipe.fit(X_train_selected,y_train)

,steps,"[('poly', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,3
,interaction_only,False
,include_bias,False
,order,'C'
,fit_intercept,True
,copy_X,True
,tol,1e-06


In [59]:
poly_pred = pipe.predict(X_test_selected)

In [60]:
poly_transformer = pipe.named_steps["poly"]

In [61]:
print("Original number of features:", X_train_selected.shape[1])
print("Polynomial feature count:", poly_transformer.n_output_features_)


Original number of features: 7
Polynomial feature count: 119


In [62]:
print('R2_score:',r2_score(y_test,poly_pred))
print('MSE:',mean_squared_error(y_test,poly_pred))

R2_score: 0.4490667281585179
MSE: 441429.70030369004


Hyperparameter tuning

In [63]:
from sklearn.model_selection import GridSearchCV

In [ ]:
pipe = Pipeline([
    ('poly', PolynomialFeatures()),
    ('model', LinearRegression())
])

para_grid = {
    'poly__degree': range(1, 8),
    'poly__include_bias': [True, False]
}

poly_grid = GridSearchCV(
    pipe,
    para_grid,
    cv=5,
    scoring='r2'
)

poly_grid.fit(X_train_selected, y_train)

print(poly_grid.best_params_)
print(poly_grid.best_score_)

In [ ]:
poly_model = poly_grid.best_estimator_

In [ ]:
poly_pred2 = poly_model.predict(X_test_selected)

In [ ]:
print('R2_score:',r2_score(y_test,poly_pred2))
print('MSE:',mean_squared_error(y_test,poly_pred2))